In [1]:
import numpy as np

# A tiny toy corpus. Each string is one "document".
# We'll convert these words into integers (integer encoding).
docs = ['go pakistan',
        'pakistan pakistan',
        'hip hip hurray',
        'jeetega bhai jeetega pakistan jeetega',
        'pakistan humesha zindabad',
        'babar babar',
        'amir amir',
        'rauf rauf',
        'pak fouj zindabad',
        'inquilab zindabad']

In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Tokenizer builds a word -> integer vocabulary from text.
# oov_token = the placeholder used at prediction time for any
# word that was NEVER seen during fitting (Out-Of-Vocabulary).
tokenizer = Tokenizer(oov_token='<nothing>')

In [4]:
# Scans all documents and assigns each unique word an integer.
# Lower integer = more frequent word (index 1 is reserved for OOV).
tokenizer.fit_on_texts(docs)

In [5]:
# word_counts: how many times each word appeared across the corpus.
tokenizer.word_counts

OrderedDict([('go', 1),
             ('pakistan', 5),
             ('hip', 2),
             ('hurray', 1),
             ('jeetega', 3),
             ('bhai', 1),
             ('humesha', 1),
             ('zindabad', 3),
             ('babar', 2),
             ('amir', 2),
             ('rauf', 2),
             ('pak', 1),
             ('fouj', 1),
             ('inquilab', 1)])

In [7]:
# document_count: total number of documents fitted (here, 10).
# document means a sentece
tokenizer.document_count

10

In [8]:
# This is INTEGER ENCODING: each sentence becomes a list of the
# integer IDs of its words. Sentences have different lengths.
sequences = tokenizer.texts_to_sequences(docs)
sequences

[[9, 2],
 [2, 2],
 [5, 5, 10],
 [3, 11, 3, 2, 3],
 [2, 12, 4],
 [6, 6],
 [7, 7],
 [8, 8],
 [13, 14, 4],
 [15, 4]]

In [9]:
from keras.utils import pad_sequences

In [10]:
# Neural networks need fixed-length inputs, but our sentences vary
# in length. Padding adds 0s so every sequence is the same length.
# padding='post' -> zeros are added at the END of each sequence.
sequences = pad_sequences(sequences, padding='post')

In [11]:
# Now every row has equal length (length = longest sentence),
# padded with trailing zeros. This is model-ready shape.
sequences

array([[ 9,  2,  0,  0,  0],
       [ 2,  2,  0,  0,  0],
       [ 5,  5, 10,  0,  0],
       [ 3, 11,  3,  2,  3],
       [ 2, 12,  4,  0,  0],
       [ 6,  6,  0,  0,  0],
       [ 7,  7,  0,  0,  0],
       [ 8,  8,  0,  0,  0],
       [13, 14,  4,  0,  0],
       [15,  4,  0,  0,  0]], dtype=int32)

## Switching to the IMDB Dataset

**Everything above was a demonstration.** The 10 cricket sentences were only
used to *show how integer encoding works* — building a vocabulary, turning
words into numbers, and padding. Those 10 sentences are now finished with and
are **not** used again; they never go into any model.

From here we use the **IMDB movie-review dataset**, which is a large, real
dataset of reviews that has **already been integer-encoded for us**. So we
skip the tokenizing/encoding step — not because it isn't needed, but because
someone did it before publishing the data. Each review arrives as a list of
word-IDs (numbers) straight away.

> In short: *the cricket sentences taught us HOW integer encoding is done;
> IMDB is a dataset where it is ALREADY done, so we just load it and build a
> model.* The one step we still do ourselves is **padding**, since the data
> comes pre-encoded but not pre-padded.

In [12]:
from keras.datasets import imdb
from keras import Sequential
from keras.layers import Dense, SimpleRNN, Embedding, Flatten

In [13]:
# IMDB movie reviews for sentiment classification (positive/negative).
# Reviews come PRE-encoded as integer sequences, where each integer
# is a word rank (1 = most frequent word in the dataset).
(X_train, y_train), (X_test, y_test) = imdb.load_data()

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [16]:
# X_train is the full set of IMDB reviews, already integer-encoded.
# Each review is just a LIST OF NUMBERS, where every number is a word's ID.
print(X_train[0])   # X_train[0] = the 1st review (a list of word-IDs)

# len() counts how many numbers (words) are inside one review.
# X_train[2] is the review at index 2 -> the 3rd review (counting starts at 0).
# The output (e.g. 141) means: THIS review is 141 words long.
# KEY POINT: different reviews have different lengths -- some short, some long.
# That is a problem, because a neural network needs every input to be the
# SAME size. Cell 14 fixes this.
len(X_train[2])

[2071   56   26  141    6  194 7486   18    4  226   22   21  134  476
   26  480    5  144   30 5535   18   51   36   28  224   92   25  104
    4  226   65   16   38 1334   88   12   16  283    5   16 4472  113
  103   32   15   16 5345   19  178   32]


50

In [18]:
# ------------------------------------------------------------------
# Understanding input_shape=(50, 1)
# An RNN reads a sequence one step at a time, so its input is described as:
#         (number_of_timesteps, features_per_timestep)
#
#   50 = timesteps          -> how many tokens are in the sequence
#                              (our padded review length: 50 tokens)
#   1  = features per token -> how many numbers describe ONE token
#
# So (50, 1) means: "a sequence of 50 steps, and at EACH step I give you
# just 1 number." That 1 number is the raw word-ID:
#
#   timestep:   1      2      3    ...   50
#   feature:  [2071]  [56]   [26]  ...  [0]
#              ^-- one plain integer per token  (this is the "1")
#
# WHY THE "1" IS THE PROBLEM:
# Describing a word with a single number forces the model to read that
# number as a QUANTITY. But word-ID 2071 is not "bigger" or "more" than
# word-ID 56 -- IDs are arbitrary labels with no real magnitude. One number
# carries almost no information, so the RNN has nothing to learn from and
# ends up guessing (~50% accuracy).
#
# THE FIX (done in the embedding version):
# Each token should be described by MANY meaningful numbers, not one. An
# Embedding layer replaces the single ID with a vector (e.g. 32 numbers),
# so features-per-timestep becomes 32 instead of a useless 1. That is why
# the embedding model REMOVES input_shape=(50, 1) entirely -- the Embedding
# layer now decides the features per timestep.
# ------------------------------------------------------------------
model = Sequential()
model.add(SimpleRNN(32, input_shape=(50, 1), return_sequences=False))
model.add(Dense(1, activation='sigmoid'))   # sigmoid -> single 0..1 output

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,121 (4.38 KB)

 Trainable params: 1,121 (4.38 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
# binary_crossentropy = correct loss for 2-class (pos/neg) problems.
# Watch val_accuracy hover around 0.50 — the model isn't learning.
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 14ms/step - accuracy: 0.5102 - loss: 0.6987 - val_accuracy: 0.5021 - val_loss: 0.6950
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.5040 - loss: 0.6933 - val_accuracy: 0.5020 - val_loss: 0.6937
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.5054 - loss: 0.6933 - val_accuracy: 0.5019 - val_loss: 0.6948
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 21s 15ms/step - accuracy: 0.5072 - loss: 0.6929 - val_accuracy: 0.5020 - val_loss: 0.6938
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.5096 - loss: 0.6924 - val_accuracy: 0.5030 - val_loss: 0.6958


## Part C — Improving the Model with Word Embedding

The baseline got stuck at ~50% accuracy (random guessing). The reason was
**not** the RNN — it was how we fed the words in. Using `input_shape=(50, 1)`
gave the model just **one plain number per word**, and a word-ID like `2071`
carries no real meaning (it is only a label, not a quantity). With nothing
meaningful to learn from, the model could not improve.

The fix is a **word embedding**, which replaces each word-ID with a **learned
vector of many numbers** that captures meaning. First we *demonstrate* what an
embedding actually does on the 10 sentences from Part A (watch integers become
vectors, and inspect the lookup table), then we apply the same idea to IMDB
with a few supporting changes:

- **Embedding layer** — each word becomes a dense vector instead of one number.
  *(This is the main fix — it removes the flawed `input_shape=(50, 1)`.)*
- **Capped vocabulary** (`num_words=10000`) — keep only the 10,000 most common words.
- **Longer sequences** (`maxlen=200`) — keep far more of each review than the 50 tokens used before.

In [22]:
# Tools we need for this embedding demo.
from tensorflow.keras.preprocessing.text import Tokenizer   # to build word -> integer vocab
from keras.utils import pad_sequences            # to make sequences equal length
from keras import Sequential                     # to stack layers into a model
from keras.layers import Embedding               # the embedding (lookup) layer

In [23]:
# The same 10 cricket sentences as Part A, but stored in a NEW variable
# (demo_docs) so this demo is fully self-contained and doesn't depend on
# anything defined earlier in the notebook.
demo_docs = ['go pakistan',
        'pakistan pakistan',
        'hip hip hurray',
        'jeetega bhai jeetega pakistan jeetega',
        'pakistan humesha zindabad',
        'babar babar',
        'amir amir',
        'rauf rauf',
        'pak fouj zindabad',
        'inquilab zindabad']

In [24]:
# Create a fresh tokenizer and let it learn a word -> integer mapping
# from our sentences. oov_token is the placeholder for unseen words.
demo_tokenizer = Tokenizer(oov_token='<nothing>')
demo_tokenizer.fit_on_texts(demo_docs)

# Peek at the learned mapping so we know which number means which word.
demo_tokenizer.word_index

{'<nothing>': 1,
 'pakistan': 2,
 'jeetega': 3,
 'zindabad': 4,
 'hip': 5,
 'babar': 6,
 'amir': 7,
 'rauf': 8,
 'go': 9,
 'hurray': 10,
 'bhai': 11,
 'humesha': 12,
 'pak': 13,
 'fouj': 14,
 'inquilab': 15}

In [25]:
# Turn each sentence into a list of integer IDs (integer encoding).
# This is the SAME step from Part A -- embedding always starts from integers.
demo_seqs = demo_tokenizer.texts_to_sequences(demo_docs)
demo_seqs

[[9, 2],
 [2, 2],
 [5, 5, 10],
 [3, 11, 3, 2, 3],
 [2, 12, 4],
 [6, 6],
 [7, 7],
 [8, 8],
 [13, 14, 4],
 [15, 4]]

In [26]:
# Sentences have different lengths, but the model needs equal-length rows.
# padding='post' adds 0s at the END until every sentence is the same length.
demo_seqs = pad_sequences(demo_seqs, padding='post')
demo_seqs

array([[ 9,  2,  0,  0,  0],
       [ 2,  2,  0,  0,  0],
       [ 5,  5, 10,  0,  0],
       [ 3, 11,  3,  2,  3],
       [ 2, 12,  4,  0,  0],
       [ 6,  6,  0,  0,  0],
       [ 7,  7,  0,  0,  0],
       [ 8,  8,  0,  0,  0],
       [13, 14,  4,  0,  0],
       [15,  4,  0,  0,  0]], dtype=int32)

In [27]:
# The Embedding layer needs to know two things:
#
# demo_vocab = how many distinct indices can appear.
#   len(word_index) counts our real words; we add +1 because index 0 is
#   reserved for the padding value, so the lookup table must include row 0.
demo_vocab = len(demo_tokenizer.word_index) + 1

# demo_len = length of each padded sentence (number of tokens per row).
# demo_seqs.shape is (10, demo_len); shape[1] grabs that length.
demo_len = demo_seqs.shape[1]

print("Vocabulary size (rows in lookup table):", demo_vocab)
print("Sequence length (tokens per sentence):", demo_len)

Vocabulary size (rows in lookup table): 16
Sequence length (tokens per sentence): 5


In [28]:
# Embedding = a trainable lookup table of shape (demo_vocab x 8).
#   input_dim=demo_vocab  -> one ROW per possible index
#   output_dim=8          -> each word is represented by an 8-number vector
#   input_length=demo_len -> how many tokens are in each input sentence
#
# Each integer ID simply selects one row of this table (its vector).
demo_model = Sequential()
demo_model.add(Embedding(input_dim=demo_vocab, output_dim=8, input_length=demo_len))

# We compile only so the model can run; we are NOT training here, just
# looking at what the embedding produces.
demo_model.compile('adam', 'mse')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [29]:
# Feed our padded integer sentences into the embedding layer.
demo_vectors = demo_model.predict(demo_seqs)

# BEFORE: shape (10, demo_len)      -> 10 sentences, each a row of integers
print("Input (integers) shape:", demo_seqs.shape)

# AFTER:  shape (10, demo_len, 8)   -> every single integer is now an
#         8-number VECTOR, so an extra dimension of size 8 appears.
print("Output (vectors) shape:", demo_vectors.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 374ms/step
Input (integers) shape: (10, 5)
Output (vectors) shape: (10, 5, 8)


In [30]:
# The first sentence: notice each token is no longer one number, but a
# list of 8 numbers. THIS is the whole point of embedding -- a word is now
# described by many meaningful values instead of a single meaningless ID.
demo_vectors[0]

array([[ 0.00709031, -0.0356069 ,  0.01674503,  0.03760953, -0.01309966,
         0.01131535, -0.04453623, -0.00544912],
       [-0.04816055,  0.01896792,  0.03555014, -0.00560538, -0.02566694,
         0.02157408, -0.00951277, -0.00447343],
       [-0.03460725,  0.00636645,  0.0257959 , -0.03507914,  0.00132217,
         0.02299077,  0.031683  , -0.02795898],
       [-0.03460725,  0.00636645,  0.0257959 , -0.03507914,  0.00132217,
         0.02299077,  0.031683  , -0.02795898],
       [-0.03460725,  0.00636645,  0.0257959 , -0.03507914,  0.00132217,
         0.02299077,  0.031683  , -0.02795898]], dtype=float32)

In [31]:
# The embedding layer's weights ARE the word vectors: a lookup table with
# ONE ROW per vocabulary index. Shape = (demo_vocab, 8), so every word (plus
# the padding slot at row 0) has its own 8-number vector.
embedding_matrix = demo_model.layers[0].get_weights()[0]
print("Embedding matrix shape:", embedding_matrix.shape)   # (demo_vocab, 8)

# Row 2 is the vector for word index 2. From demo_tokenizer.word_index that
# index is 'india'. During real TRAINING, rows for words used in similar
# contexts drift closer together -- something a single integer ID can never do.
embedding_matrix[2]

Embedding matrix shape: (16, 8)


array([-0.04816055,  0.01896792,  0.03555014, -0.00560538, -0.02566694,
        0.02157408, -0.00951277, -0.00447343], dtype=float32)

### Applying Embedding to the IMDB Model

The demo above showed *what* an embedding does on 10 sentences. Now we use the
exact same idea on the real IMDB task: instead of feeding raw integer IDs into
the RNN (the flawed baseline with `input_shape=(50, 1)`), we put an Embedding
layer first, so the RNN receives meaningful learned vectors — many numbers per
word instead of one.

In [32]:
# Imports needed for the IMDB model (included here so this section is
# self-contained and you don't have to scroll back up).
from keras.datasets import imdb
from keras.utils import pad_sequences
from keras import Sequential
from keras.layers import Embedding, SimpleRNN, Dense

# num_words=10000 keeps only the 10,000 most frequent words; rarer words are
# dropped as noise. Reviews still arrive already integer-encoded.
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

In [33]:
# maxlen=200 keeps far more of each review than the 50 used in the baseline,
# so much less information is thrown away (recall a 141-word review lost most
# of its words at maxlen=50).
maxlen = 200
X_train = pad_sequences(X_train, padding='post', maxlen=maxlen)
X_test  = pad_sequences(X_test,  padding='post', maxlen=maxlen)

In [34]:
# KEY CHANGE from the baseline: there is NO input_shape=(50, 1) here.
# The Embedding layer turns each integer ID into a learned 32-dim vector,
# so the RNN now gets 32 meaningful numbers per word instead of 1 useless one.
#   input_dim=vocab_size  -> one row per possible word index
#   output_dim=32         -> each word becomes a 32-number vector
#   input_length=maxlen   -> tokens per review
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=32, input_length=maxlen))
model.add(SimpleRNN(64))                     # reads the sequence of vectors
model.add(Dense(1, activation='sigmoid'))    # single 0..1 sentiment output
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [35]:
# Watch val_accuracy here vs. the ~0.50 baseline -- with embedding the model
# can actually learn sentiment, so accuracy rises well above chance.
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=5, batch_size=128,
          validation_data=(X_test, y_test))

Epoch 1/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 25s 114ms/step - accuracy: 0.5189 - loss: 0.6902 - val_accuracy: 0.5360 - val_loss: 0.6825
Epoch 2/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 20s 102ms/step - accuracy: 0.6074 - loss: 0.6465 - val_accuracy: 0.5464 - val_loss: 0.6769
Epoch 3/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 22s 108ms/step - accuracy: 0.6576 - loss: 0.5720 - val_accuracy: 0.5342 - val_loss: 0.7255
Epoch 4/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 42s 113ms/step - accuracy: 0.7169 - loss: 0.4636 - val_accuracy: 0.5374 - val_loss: 0.7920
Epoch 5/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 39s 102ms/step - accuracy: 0.7512 - loss: 0.3971 - val_accuracy: 0.5425 - val_loss: 0.8396


In [36]:
# Final accuracy on the unseen test set.
loss, acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {acc:.4f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.5425 - loss: 0.8396
Test accuracy: 0.5425
